In [1]:
import os
import torch
from google.colab import drive

drive.mount('/content/drive')

!pip install -q imageio[ffmpeg]
!pip install -q matplotlib
!pip install -q opencv-python
!pip install -q torch torchvision
!pip install -q git+https://github.com/facebookresearch/co-tracker.git

print("⬇️ Downloading CoTracker3 Weights...")
if not os.path.exists("checkpoints"):
    os.makedirs("checkpoints", exist_ok=True)

!wget -q -O checkpoints/scaled_offline.pth https://huggingface.co/facebook/cotracker3/resolve/main/scaled_offline.pth

print("\n✅ READY! Proceed to Block 2.")

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import os
import glob
import torch
import gc
import numpy as np
import torch.nn.functional as F
from cotracker.predictor import CoTrackerPredictor
from cotracker.utils.visualizer import Visualizer, read_video_from_path

# === SETTINGS ===
base_input_dir = "/content/drive/MyDrive/EgoDex_Data/input_video/video_learning_samples"
base_output_dir = "/content/drive/MyDrive/EgoDex_Data/cotracker_output"

grid_size = 30         # Reduced from 50 to 30 (900 points vs 2500 points)
MAX_SHORT_SIDE = 480   # Keep quality decent (480p)
MAX_FRAMES = 150       # Prevents T^2 memory explosion.
# ================

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🚀 Loading CoTracker3 Model on {device}...")

model = CoTrackerPredictor(checkpoint=os.path.join("checkpoints", "scaled_offline.pth"))
model = model.eval()
model = model.to(device)

subdirs = [f.path for f in os.scandir(base_input_dir) if f.is_dir()]
print(f"📂 Found {len(subdirs)} categories to process.")

for folder_path in subdirs:
    folder_name = os.path.basename(folder_path)

    video_files = glob.glob(os.path.join(folder_path, "*.mp4"))
    if not video_files:
        continue

    target_video_path = video_files[0]
    video_filename = os.path.basename(target_video_path)

    save_dir = os.path.join(base_output_dir, folder_name)
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, video_filename.replace(".mp4", "_tracked.mp4"))

    if os.path.exists(save_path):
        print(f"⏩ Skipping {folder_name} (Already done)")
        continue

    print(f"\n▶️ Processing: {folder_name} -> {video_filename}")

    # Initialize cleanup vars
    video = None
    pred_tracks = None
    pred_visibility = None

    try:
        video = read_video_from_path(target_video_path)

        if isinstance(video, np.ndarray):
            video = torch.from_numpy(video)

        if video.ndim == 4:
            video = video.permute(0, 3, 1, 2)[None].float()

        _, T, C, H, W = video.shape
        if min(H, W) > MAX_SHORT_SIDE:
            scale_factor = MAX_SHORT_SIDE / min(H, W)
            new_H, new_W = int(H * scale_factor), int(W * scale_factor)

            video_reshaped = video.view(-1, C, H, W)
            video_reshaped = F.interpolate(video_reshaped, size=(new_H, new_W), mode='bilinear', align_corners=False)
            video = video_reshaped.view(1, T, C, new_H, new_W)
            print(f"   📉 Resized video to {new_W}x{new_H}")

        if T > MAX_FRAMES:
            print(f"   ✂️  Video too long ({T} frames). Truncating to first {MAX_FRAMES} frames...")
            video = video[:, :MAX_FRAMES]

        video = video.to(device)

        with torch.no_grad():
            pred_tracks, pred_visibility = model(video, grid_size=grid_size)

        vis = Visualizer(save_dir=save_dir, pad_value=100, linewidth=2)
        vis.visualize(video, pred_tracks, pred_visibility, filename=video_filename.replace(".mp4", ""))
        print(f"   ✅ Saved to: {save_path}")

    except torch.cuda.OutOfMemoryError:
        print("   ❌ Failed: Even with truncation, it was too big. (Try lowering MAX_FRAMES to 80)")
    except Exception as e:
        print(f"   ❌ Failed: {e}")
    finally:
        del video, pred_tracks, pred_visibility
        gc.collect()
        torch.cuda.empty_cache()

print("\n🎉 ALL CATEGORIES COMPLETE!")

🚀 Loading CoTracker3 Model on cuda...
📂 Found 5 categories to process.

▶️ Processing: open_close -> 1.mp4
   📉 Resized video to 853x480
   ✂️  Video too long (171 frames). Truncating to first 150 frames...


Video saved to /content/drive/MyDrive/EgoDex_Data/cotracker_output/open_close/1.mp4
   ✅ Saved to: /content/drive/MyDrive/EgoDex_Data/cotracker_output/open_close/1_tracked.mp4

▶️ Processing: add_remove_lid -> 0.mp4
   📉 Resized video to 853x480


Video saved to /content/drive/MyDrive/EgoDex_Data/cotracker_output/add_remove_lid/0.mp4
   ✅ Saved to: /content/drive/MyDrive/EgoDex_Data/cotracker_output/add_remove_lid/0_tracked.mp4

▶️ Processing: basic_pick_and_place -> 1.mp4
   📉 Resized video to 853x480
   ✂️  Video too long (160 frames). Truncating to first 150 frames...


Video saved to /content/drive/MyDrive/EgoDex_Data/cotracker_output/basic_pick_and_place/1.mp4
   ✅ Saved to: /content/drive/MyDrive/EgoDex_Data/cotracker_output/basic_pick_and_place/1_tracked.mp4

▶️ Processing: assemble_disassemble_furniture_bench_stool -> 14.mp4
   📉 Resized video to 853x480


Video saved to /content/drive/MyDrive/EgoDex_Data/cotracker_output/assemble_disassemble_furniture_bench_stool/14.mp4
   ✅ Saved to: /content/drive/MyDrive/EgoDex_Data/cotracker_output/assemble_disassemble_furniture_bench_stool/14_tracked.mp4

▶️ Processing: insert_remove -> 3.mp4
   📉 Resized video to 853x480
   ✂️  Video too long (203 frames). Truncating to first 150 frames...


Video saved to /content/drive/MyDrive/EgoDex_Data/cotracker_output/insert_remove/3.mp4
   ✅ Saved to: /content/drive/MyDrive/EgoDex_Data/cotracker_output/insert_remove/3_tracked.mp4

🎉 ALL CATEGORIES COMPLETE!
